# Temporary: raw images + ink crop, grayscale (no black-and-white)

Same as `kaggle_raw_run.ipynb`, but it also applies the **ink-crop** step while keeping
the image **grayscale**. Compared to the other notebook:

| notebook | ink crop | background whitening | binarization (0/255) |
|---|---|---|---|
| `kaggle_raw_run.ipynb` (`--resize-only`) | no | no | no |
| **this notebook** (`--crop-grayscale`) | **yes** | no | no |
| original `data/processed` pipeline | yes | yes | yes |

`--crop-grayscale` = grayscale -> crop to the ink bounding box -> resize to 192x384.
Strokes keep their original intensity (pressure / ink flow) and the paper background is
left as-is, so the input is not black-and-white.

**Before running:** upload `sigver_raw.zip` as a Kaggle Dataset (it extracts to
`manifest.csv` + `raw/...`), enable a GPU accelerator, and turn on Internet.

In [ ]:
# --- Setup: code + dependencies ---------------------------------------------
%cd /kaggle/working
!rm -rf sigver && git clone <your-repo-url> sigver
%cd /kaggle/working/sigver
!pip install -q timm opencv-python-headless matplotlib

# Raw .tif scans make OpenCV print a harmless "TIFF Software tag" warning per file.
import os
os.environ["OPENCV_LOG_LEVEL"] = "ERROR"

In [ ]:
# --- Paths -------------------------------------------------------------------
DATA = "/kaggle/input/sigver-raw"          # dataset root: manifest.csv + raw/
RUN = "ckpt_crop_gray"                      # checkpoint folder under /kaggle/working
CKPT = f"/kaggle/working/{RUN}/best_model.pt"
LOG = "/kaggle/working/training_log_crop.txt"
CURVES = "/kaggle/working/curves_crop.png"
EVAL_JSON = "/kaggle/working/eval_default_crop.json"

!ls $DATA && head -2 $DATA/manifest.csv

In [ ]:
# --- Train on RAW images with the ink crop, grayscale (no black-and-white) ----
# --crop-grayscale : grayscale, crop to the ink box, resize to 192x384
#                    (no background whitening, no binarization)
# --negatives auto : EER/accuracy vs skilled forgeries (present in this dataset)
!python -m training.train_siamese --manifest $DATA/manifest.csv \
  --crop-grayscale --negatives auto \
  --output-dir /kaggle/working/{RUN} \
  --device cuda --workers 2 \
  --phase1-epochs 5 --phase2-epochs 30 --lr-schedule cosine --select-by eer \
  2>&1 | tee {LOG}

In [ ]:
# --- Training curves ---------------------------------------------------------
!python -m training.analyze_log {LOG} --plot {CURVES}

from IPython.display import Image, display
display(Image(CURVES))

In [ ]:
# --- Evaluate on the held-out test signers -----------------------------------
# Threshold is chosen on the validation signers (EER) and applied to test.
!python -m training.evaluate --checkpoint {CKPT} --manifest $DATA/manifest.csv \
  --crop-grayscale --negatives auto \
  --device cuda --workers 2 --bootstrap 1000 \
  2>&1 | tee {EVAL_JSON}

In [ ]:
# --- Accuracy at fixed similarity thresholds (bootstrap off for speed) -------
for t in [0.7, 0.8, 0.85, 0.9]:
    print("=== threshold", t)
    !python -m training.evaluate --checkpoint {CKPT} --manifest $DATA/manifest.csv \
      --crop-grayscale --negatives auto \
      --device cuda --workers 2 --bootstrap 0 --threshold {t} \
      2>&1 | grep -E "threshold_used|accuracy_skilled|far_skilled_at_threshold|frr_at_threshold" | head -5

In [ ]:
# --- Saved checkpoint --------------------------------------------------------
!ls -lh {CKPT}

### Notes

- **Saving the model:** use **Save Version -> Save & Run All (Commit)** so the run
  survives closing the tab, then download `best_model.pt` from the Output tab.
- **Comparing experiments:** run `kaggle_raw_run.ipynb` (`--resize-only`) and this one
  (`--crop-grayscale`) and compare `validation_eer` / `accuracy` / `balanced_accuracy`.
  If the crop helps a lot, the crop step is what mattered; whitening and binarization
  can then be tested separately.
- **Resize mode:** `--resize-mode stretch` forces 192x384 instead of the default
  aspect-preserving resize with white padding.
- **Negatives:** `--negatives auto` uses skilled forgeries when present (this dataset
  has them) and falls back to random/different-signer pairs otherwise.